In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("📚 Libraries imported successfully!")
print("📁 Current working directory:", Path.cwd())


📚 Libraries imported successfully!
📁 Current working directory: /Users/tushartyagi/Documents/debiased_ranking


In [2]:
def load_evaluation_checkpoints():
    """
    Load all evaluation checkpoint files and organize them by dataset type
    """
    print("🔍 Scanning for evaluation checkpoint files...")
    
    # Find all evaluation checkpoint JSON files
    checkpoint_files = glob.glob("evaluation_checkpoint*.json")
    
    checkpoints = {}
    for file_path in checkpoint_files:
        print(f"📁 Loading: {file_path}")
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
            
            # Extract dataset type from filename
            if 'movielens' in file_path:
                dataset_type = 'movielens'
            elif 'music' in file_path:
                dataset_type = 'music'
            elif 'books' in file_path:
                dataset_type = 'books'
            elif 'news' in file_path:
                dataset_type = 'news'
            else:
                dataset_type = 'unknown'
            
            # Determine experiment variant
            variant = file_path.replace('evaluation_checkpoint_', '').replace('.json', '')
            
            if dataset_type not in checkpoints:
                checkpoints[dataset_type] = {}
            
            checkpoints[dataset_type][variant] = {
                'file_path': file_path,
                'data': data
            }
            
        except Exception as e:
            print(f"❌ Error loading {file_path}: {e}")
    
    print(f"\n✅ Successfully loaded {len(checkpoint_files)} checkpoint files")
    
    # Print summary
    for dataset, variants in checkpoints.items():
        print(f"📊 {dataset.upper()}: {len(variants)} variants")
        for variant in variants.keys():
            print(f"   - {variant}")
    
    return checkpoints

# Load all checkpoints
checkpoints = load_evaluation_checkpoints()


🔍 Scanning for evaluation checkpoint files...
📁 Loading: evaluation_checkpoint_bias20_music.json
📁 Loading: evaluation_checkpoint_bias20_movielens copy.json
📁 Loading: evaluation_checkpoint_news7.json
📁 Loading: evaluation_checkpoint_bias20_music_11.json
📁 Loading: evaluation_checkpoint_bias20_music_10.json
📁 Loading: evaluation_checkpoint_bias20_music_1.json
📁 Loading: evaluation_checkpoint_books.json
📁 Loading: evaluation_checkpoint_news6.json
📁 Loading: evaluation_checkpoint_news5.json
❌ Error loading evaluation_checkpoint_news5.json: Expecting ',' delimiter: line 425608 column 13 (char 18992324)
📁 Loading: evaluation_checkpoint_bias20_movielens.json
📁 Loading: evaluation_checkpoint_bias20_music_2.json
📁 Loading: evaluation_checkpoint_bias20_music_3.json
📁 Loading: evaluation_checkpoint_news8.json
📁 Loading: evaluation_checkpoint_bias20_music_4.json
📁 Loading: evaluation_checkpoint_news3.json
📁 Loading: evaluation_checkpoint_news.json

✅ Successfully loaded 16 checkpoint files
📊 MUS

In [3]:
def extract_debiased_scores_and_rankings(checkpoint_data):
    """
    Extract debiased scores and create rankings from checkpoint data
    Returns a list of rankings for Borda count analysis
    """
    rankings_data = []
    
    # Check if this is user evaluation data
    if 'evaluation_results' in checkpoint_data:
        eval_results = checkpoint_data['evaluation_results']
        print(f"📊 Found evaluation results with {eval_results.get('accuracy', {}).get('num_evaluations', 0)} evaluations")
        return rankings_data
    
    # Check for individual user results
    if 'user_results' in checkpoint_data:
        user_results = checkpoint_data['user_results']
        print(f"👥 Processing {len(user_results)} user results...")
        
        for user_result in user_results:
            if 'debiased_scores' in user_result and user_result['debiased_scores']:
                # Extract debiased scores
                debiased_scores = user_result['debiased_scores']
                
                # Create ranking based on debiased scores
                items_with_scores = [(item, score) for item, score in debiased_scores.items()]
                # Sort by debiased score (descending)
                items_with_scores.sort(key=lambda x: x[1], reverse=True)
                
                ranking = [item for item, score in items_with_scores]
                rankings_data.append({
                    'user_id': user_result.get('user_id', 'unknown'),
                    'ranking': ranking,
                    'scores': debiased_scores,
                    'target_item': user_result.get('target_item', 'unknown')
                })
    
    # Check for recomputed user results  
    elif 'recomputed_user_results' in checkpoint_data:
        user_results = checkpoint_data['recomputed_user_results']
        print(f"👥 Processing {len(user_results)} recomputed user results...")
        
        for user_result in user_results:
            if 'debiased_scores' in user_result and user_result['debiased_scores']:
                debiased_scores = user_result['debiased_scores']
                items_with_scores = [(item, score) for item, score in debiased_scores.items()]
                items_with_scores.sort(key=lambda x: x[1], reverse=True)
                
                ranking = [item for item, score in items_with_scores]
                rankings_data.append({
                    'user_id': user_result.get('user_id', 'unknown'),
                    'ranking': ranking,
                    'scores': debiased_scores,
                    'target_item': user_result.get('target_item', 'unknown')
                })
    
    print(f"✅ Extracted {len(rankings_data)} rankings with debiased scores")
    return rankings_data

# Test extraction on first available checkpoint
test_dataset = list(checkpoints.keys())[0] if checkpoints else None
if test_dataset:
    test_variant = list(checkpoints[test_dataset].keys())[0]
    test_data = checkpoints[test_dataset][test_variant]['data']
    test_rankings = extract_debiased_scores_and_rankings(test_data)
    print(f"\n🧪 Test extraction on {test_dataset} - {test_variant}:")
    if test_rankings:
        print(f"   First ranking sample: {test_rankings[0]['ranking'][:5]}...")
        print(f"   Target item: {test_rankings[0]['target_item']}")
    else:
        print("   No rankings found in test data")
else:
    print("⚠️ No checkpoints available for testing")


✅ Extracted 0 rankings with debiased scores

🧪 Test extraction on music - bias20_music:
   No rankings found in test data


In [4]:
def borda_count_scoring(rankings_list, normalize=True):
    """
    Implement Borda count method (Emerson 2013) on multiple rankings
    
    Args:
        rankings_list: List of rankings, where each ranking is a list of items
        normalize: Whether to normalize scores by number of rankings
    
    Returns:
        Dictionary with items as keys and Borda count scores as values
    """
    print(f"🗳️ Applying Borda count to {len(rankings_list)} rankings...")
    
    borda_scores = defaultdict(int)
    item_appearances = defaultdict(int)
    
    for ranking in rankings_list:
        n_items = len(ranking)
        
        # Assign Borda count scores: first position gets n points, second gets n-1, etc.
        for position, item in enumerate(ranking):
            score = n_items - position  # First position (0) gets n points
            borda_scores[item] += score
            item_appearances[item] += 1
    
    # Convert to regular dict and optionally normalize
    final_scores = dict(borda_scores)
    
    if normalize:
        # Normalize by number of rankings that included each item
        for item in final_scores:
            if item_appearances[item] > 0:
                final_scores[item] = final_scores[item] / item_appearances[item]
    
    print(f"✅ Computed Borda scores for {len(final_scores)} unique items")
    return final_scores, dict(item_appearances)

def analyze_borda_count_results(borda_scores, item_appearances, top_k=20):
    """
    Analyze and summarize Borda count results
    """
    print(f"\n📊 BORDA COUNT ANALYSIS RESULTS")
    print("=" * 50)
    
    # Sort items by Borda score
    sorted_items = sorted(borda_scores.items(), key=lambda x: x[1], reverse=True)
    
    print(f"📈 Total items analyzed: {len(sorted_items)}")
    print(f"🏆 Top {min(top_k, len(sorted_items))} items by Borda count:")
    print()
    
    for i, (item, score) in enumerate(sorted_items[:top_k], 1):
        appearances = item_appearances[item]
        print(f"{i:2d}. {item:<50} Score: {score:8.3f} (appeared in {appearances} rankings)")
    
    # Statistics
    scores = list(borda_scores.values())
    appearances = list(item_appearances.values())
    
    print(f"\n📊 SCORE STATISTICS:")
    print(f"   Mean score: {np.mean(scores):.3f}")
    print(f"   Median score: {np.median(scores):.3f}")
    print(f"   Std deviation: {np.std(scores):.3f}")
    print(f"   Min score: {np.min(scores):.3f}")
    print(f"   Max score: {np.max(scores):.3f}")
    
    print(f"\n👥 APPEARANCE STATISTICS:")
    print(f"   Mean appearances: {np.mean(appearances):.1f}")
    print(f"   Median appearances: {np.median(appearances):.1f}")
    print(f"   Max appearances: {np.max(appearances)}")
    print(f"   Min appearances: {np.min(appearances)}")
    
    return sorted_items

# Test Borda count on sample data
print("🧪 Testing Borda count implementation with sample data...")

# Create sample rankings for testing
sample_rankings = [
    ['A', 'B', 'C', 'D'],
    ['B', 'A', 'D', 'C'],
    ['A', 'C', 'B', 'D'],
    ['C', 'A', 'B', 'D']
]

sample_scores, sample_appearances = borda_count_scoring(sample_rankings)
print("\n📋 Sample Borda count results:")
for item, score in sorted(sample_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"   {item}: {score:.2f} points (appeared {sample_appearances[item]} times)")

print("\n✅ Borda count implementation working correctly!")


🧪 Testing Borda count implementation with sample data...
🗳️ Applying Borda count to 4 rankings...
✅ Computed Borda scores for 4 unique items

📋 Sample Borda count results:
   A: 3.50 points (appeared 4 times)
   B: 2.75 points (appeared 4 times)
   C: 2.50 points (appeared 4 times)
   D: 1.25 points (appeared 4 times)

✅ Borda count implementation working correctly!


In [5]:
def process_all_checkpoints_with_borda_count(checkpoints):
    """
    Process all evaluation checkpoints and apply Borda count analysis
    """
    print("🔄 PROCESSING ALL CHECKPOINTS WITH BORDA COUNT ANALYSIS")
    print("=" * 60)
    
    all_results = {}
    
    for dataset_type, variants in checkpoints.items():
        print(f"\n📊 Processing {dataset_type.upper()} dataset...")
        dataset_results = {}
        
        for variant_name, variant_data in variants.items():
            print(f"\n🔍 Analyzing variant: {variant_name}")
            
            # Extract rankings from this checkpoint
            rankings_data = extract_debiased_scores_and_rankings(variant_data['data'])
            
            if not rankings_data:
                print(f"   ⚠️ No rankings found in {variant_name}")
                continue
            
            # Extract just the rankings for Borda count
            rankings_list = [rd['ranking'] for rd in rankings_data if rd['ranking']]
            
            if not rankings_list:
                print(f"   ⚠️ No valid rankings found in {variant_name}")
                continue
            
            # Apply Borda count
            borda_scores, item_appearances = borda_count_scoring(rankings_list, normalize=True)
            
            # Analyze results
            sorted_items = analyze_borda_count_results(borda_scores, item_appearances, top_k=10)
            
            # Store results
            dataset_results[variant_name] = {
                'borda_scores': borda_scores,
                'item_appearances': item_appearances,
                'sorted_items': sorted_items,
                'num_rankings': len(rankings_list),
                'num_unique_items': len(borda_scores),
                'rankings_data': rankings_data
            }
        
        all_results[dataset_type] = dataset_results
    
    return all_results

# Process all checkpoints
print("🚀 Starting comprehensive Borda count analysis...")
borda_results = process_all_checkpoints_with_borda_count(checkpoints)


🚀 Starting comprehensive Borda count analysis...
🔄 PROCESSING ALL CHECKPOINTS WITH BORDA COUNT ANALYSIS

📊 Processing MUSIC dataset...

🔍 Analyzing variant: bias20_music
✅ Extracted 0 rankings with debiased scores
   ⚠️ No rankings found in bias20_music

🔍 Analyzing variant: bias20_music_11
✅ Extracted 0 rankings with debiased scores
   ⚠️ No rankings found in bias20_music_11

🔍 Analyzing variant: bias20_music_10
✅ Extracted 0 rankings with debiased scores
   ⚠️ No rankings found in bias20_music_10

🔍 Analyzing variant: bias20_music_1
✅ Extracted 0 rankings with debiased scores
   ⚠️ No rankings found in bias20_music_1

🔍 Analyzing variant: bias20_music_2
✅ Extracted 0 rankings with debiased scores
   ⚠️ No rankings found in bias20_music_2

🔍 Analyzing variant: bias20_music_3
✅ Extracted 0 rankings with debiased scores
   ⚠️ No rankings found in bias20_music_3

🔍 Analyzing variant: bias20_music_4
✅ Extracted 0 rankings with debiased scores
   ⚠️ No rankings found in bias20_music_4

📊 P

In [ ]:
def create_borda_count_visualizations(borda_results):
    """
    Create comprehensive visualizations of Borda count results
    """
    print("📊 Creating Borda count visualizations...")
    
    # Create subplots for different visualizations
    n_datasets = len(borda_results)
    if n_datasets == 0:
        print("⚠️ No data available for visualization")
        return
    
    # 1. Top items comparison across datasets
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Borda Count Analysis Results Across Datasets', fontsize=16, fontweight='bold')
    
    dataset_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
    
    # Plot 1: Top 10 items per dataset (if multiple variants, use first one)
    ax1 = axes[0, 0]
    dataset_idx = 0
    
    for dataset_type, variants in borda_results.items():
        if not variants:
            continue
            
        # Use first variant for this visualization
        variant_name = list(variants.keys())[0]
        variant_data = variants[variant_name]
        
        top_items = variant_data['sorted_items'][:10]
        items = [item[0][:20] + '...' if len(item[0]) > 20 else item[0] for item in top_items]  # Truncate long names
        scores = [item[1] for item in top_items]
        
        y_pos = np.arange(len(items))
        color = dataset_colors[dataset_idx % len(dataset_colors)]
        
        ax1.barh(y_pos, scores, alpha=0.7, color=color, label=f'{dataset_type.title()}')
        
        dataset_idx += 1
    
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(items)
    ax1.set_xlabel('Borda Count Score')
    ax1.set_title('Top 10 Items by Borda Count')
    ax1.legend()
    ax1.grid(axis='x', alpha=0.3)
    
    # Plot 2: Score distribution
    ax2 = axes[0, 1]
    dataset_idx = 0
    
    for dataset_type, variants in borda_results.items():
        if not variants:
            continue
            
        variant_name = list(variants.keys())[0]
        variant_data = variants[variant_name]
        
        scores = list(variant_data['borda_scores'].values())
        color = dataset_colors[dataset_idx % len(dataset_colors)]
        
        ax2.hist(scores, bins=30, alpha=0.6, color=color, label=f'{dataset_type.title()}', density=True)
        dataset_idx += 1
    
    ax2.set_xlabel('Borda Count Score')
    ax2.set_ylabel('Density')
    ax2.set_title('Distribution of Borda Count Scores')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    # Plot 3: Number of rankings vs unique items
    ax3 = axes[1, 0]
    dataset_names = []
    num_rankings = []
    num_unique_items = []
    
    for dataset_type, variants in borda_results.items():
        for variant_name, variant_data in variants.items():
            dataset_names.append(f"{dataset_type}\n{variant_name}")
            num_rankings.append(variant_data['num_rankings'])
            num_unique_items.append(variant_data['num_unique_items'])
    
    x_pos = np.arange(len(dataset_names))
    width = 0.35
    
    bars1 = ax3.bar(x_pos - width/2, num_rankings, width, label='Number of Rankings', alpha=0.8, color='#FF6B6B')
    bars2 = ax3.bar(x_pos + width/2, num_unique_items, width, label='Unique Items', alpha=0.8, color='#4ECDC4')
    
    ax3.set_xlabel('Dataset Variants')
    ax3.set_ylabel('Count')
    ax3.set_title('Rankings vs Unique Items by Dataset')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(dataset_names, rotation=45, ha='right')
    ax3.legend()
    ax3.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax3.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)
    
    for bar in bars2:
        height = bar.get_height()
        ax3.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)
    
    # Plot 4: Item appearance frequency
    ax4 = axes[1, 1]
    
    all_appearances = []
    for dataset_type, variants in borda_results.items():
        for variant_name, variant_data in variants.items():
            appearances = list(variant_data['item_appearances'].values())
            all_appearances.extend(appearances)
    
    if all_appearances:
        ax4.hist(all_appearances, bins=20, alpha=0.7, color='#96CEB4', edgecolor='black')
        ax4.set_xlabel('Number of Appearances')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Distribution of Item Appearances Across Rankings')
        ax4.grid(alpha=0.3)
        
        # Add statistics
        mean_app = np.mean(all_appearances)
        median_app = np.median(all_appearances)
        ax4.axvline(mean_app, color='red', linestyle='--', alpha=0.8, label=f'Mean: {mean_app:.1f}')
        ax4.axvline(median_app, color='orange', linestyle='--', alpha=0.8, label=f'Median: {median_app:.1f}')
        ax4.legend()
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Create visualizations
if borda_results:
    viz_fig = create_borda_count_visualizations(borda_results)
else:
    print("⚠️ No Borda count results available for visualization")


In [ ]:
def generate_comprehensive_summary(borda_results):
    """
    Generate a comprehensive summary of all Borda count results
    """
    print("\n" + "="*80)
    print("🎯 COMPREHENSIVE BORDA COUNT ANALYSIS SUMMARY")
    print("="*80)
    
    summary_stats = {}
    
    for dataset_type, variants in borda_results.items():
        print(f"\n📊 {dataset_type.upper()} DATASET SUMMARY:")
        print("-" * 50)
        
        dataset_summary = {}
        
        for variant_name, variant_data in variants.items():
            print(f"\n🔍 Variant: {variant_name}")
            print(f"   📈 Total rankings processed: {variant_data['num_rankings']}")
            print(f"   🎯 Unique items found: {variant_data['num_unique_items']}")
            
            # Top 5 items
            top_5 = variant_data['sorted_items'][:5]
            print(f"   🏆 Top 5 items by Borda count:")
            for i, (item, score) in enumerate(top_5, 1):
                appearances = variant_data['item_appearances'][item]
                print(f"      {i}. {item[:40]:<40} (Score: {score:.3f}, Apps: {appearances})")
            
            # Statistics
            scores = list(variant_data['borda_scores'].values())
            appearances = list(variant_data['item_appearances'].values())
            
            stats = {
                'num_rankings': variant_data['num_rankings'],
                'num_unique_items': variant_data['num_unique_items'],
                'mean_score': np.mean(scores),
                'median_score': np.median(scores),
                'max_score': np.max(scores),
                'mean_appearances': np.mean(appearances),
                'max_appearances': np.max(appearances)
            }
            
            dataset_summary[variant_name] = stats
            
        summary_stats[dataset_type] = dataset_summary
    
    return summary_stats

def export_borda_results(borda_results, summary_stats):
    """
    Export Borda count results to files for further analysis
    """
    print(f"\n💾 EXPORTING RESULTS")
    print("-" * 30)
    
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    
    # Export detailed results
    detailed_results = {}
    for dataset_type, variants in borda_results.items():
        detailed_results[dataset_type] = {}
        for variant_name, variant_data in variants.items():
            detailed_results[dataset_type][variant_name] = {
                'borda_scores': variant_data['borda_scores'],
                'item_appearances': variant_data['item_appearances'],
                'top_20_items': variant_data['sorted_items'][:20],
                'statistics': {
                    'num_rankings': variant_data['num_rankings'],
                    'num_unique_items': variant_data['num_unique_items'],
                    'mean_score': float(np.mean(list(variant_data['borda_scores'].values()))),
                    'median_score': float(np.median(list(variant_data['borda_scores'].values()))),
                    'max_score': float(np.max(list(variant_data['borda_scores'].values())))
                }
            }
    
    detailed_filename = f"borda_count_detailed_results_{timestamp}.json"
    with open(detailed_filename, 'w') as f:
        json.dump(detailed_results, f, indent=2)
    print(f"✅ Detailed results exported to: {detailed_filename}")
    
    # Export summary statistics
    summary_filename = f"borda_count_summary_{timestamp}.json"
    with open(summary_filename, 'w') as f:
        json.dump(summary_stats, f, indent=2)
    print(f"✅ Summary statistics exported to: {summary_filename}")
    
    # Create CSV for top items across all datasets
    top_items_data = []
    for dataset_type, variants in borda_results.items():
        for variant_name, variant_data in variants.items():
            for rank, (item, score) in enumerate(variant_data['sorted_items'][:50], 1):
                appearances = variant_data['item_appearances'][item]
                top_items_data.append({
                    'dataset': dataset_type,
                    'variant': variant_name,
                    'rank': rank,
                    'item': item,
                    'borda_score': score,
                    'appearances': appearances
                })
    
    if top_items_data:
        df = pd.DataFrame(top_items_data)
        csv_filename = f"borda_count_top_items_{timestamp}.csv"
        df.to_csv(csv_filename, index=False)
        print(f"✅ Top items CSV exported to: {csv_filename}")
        print(f"   📊 CSV contains {len(df)} rows across {len(df['dataset'].unique())} datasets")
    
    return detailed_filename, summary_filename, csv_filename if top_items_data else None

# Generate comprehensive summary
if borda_results:
    summary_stats = generate_comprehensive_summary(borda_results)
    
    # Export results
    detailed_file, summary_file, csv_file = export_borda_results(borda_results, summary_stats)
    
    print(f"\n🎉 ANALYSIS COMPLETE!")
    print(f"📁 Generated files:")
    print(f"   - {detailed_file}")
    print(f"   - {summary_file}")
    if csv_file:
        print(f"   - {csv_file}")
    
    print(f"\n📋 FINAL SUMMARY:")
    total_datasets = len(borda_results)
    total_variants = sum(len(variants) for variants in borda_results.values())
    total_rankings = sum(
        variant_data['num_rankings'] 
        for variants in borda_results.values() 
        for variant_data in variants.values()
    )
    total_unique_items = sum(
        variant_data['num_unique_items'] 
        for variants in borda_results.values() 
        for variant_data in variants.values()
    )
    
    print(f"   🎯 Datasets analyzed: {total_datasets}")
    print(f"   🔄 Variants processed: {total_variants}")
    print(f"   📊 Total rankings: {total_rankings}")
    print(f"   🎨 Total unique items: {total_unique_items}")
    
else:
    print("⚠️ No Borda count results available for summary and export")
